In [1]:
import re
import sqlite3
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

## Load and Inspect
RAW = "D:/My Resume/IBM_TELECOM/telcom/Telco_Customer_Churn.xlsx"
CLEAN_CSV = "D:/My Resume/IBM_TELECOM/telcom/telco_clean.csv"
DB = "D:/My Resume/IBM_TELECOM/telcom/churn.db"


# 1. Load and Inspect

In [2]:
df = pd.read_excel(RAW)
print("Shape:", df.shape)
print(df.dtypes)
print(df.head())

Shape: (7043, 33)
CustomerID            object
Count                  int64
Country               object
State                 object
City                  object
Zip Code               int64
Lat Long              object
Latitude             float64
Longitude            float64
Gender                object
Senior Citizen        object
Partner               object
Dependents            object
Tenure Months          int64
Phone Service         object
Multiple Lines        object
Internet Service      object
Online Security       object
Online Backup         object
Device Protection     object
Tech Support          object
Streaming TV          object
Streaming Movies      object
Contract              object
Paperless Billing     object
Payment Method        object
Monthly Charges      float64
Total Charges         object
Churn Label           object
Churn Value            int64
Churn Score            int64
CLTV                   int64
Churn Reason          object
dtype: object
   Customer

In [3]:
df.head()

,CustomerID,Count,Country,State,City,Zip Code,Lat Long,Latitude,Longitude,Gender,...,Contract,Paperless Billing,Payment Method,Monthly Charges,Total Charges,Churn Label,Churn Value,Churn Score,CLTV,Churn Reason
0,3668-QPYBK,1,United States,California,Los Angeles,90003,"33.964131, -118.272783",33.964131,-118.272783,Male,...,Month-to-month,Yes,Mailed check,53.85,108.15,Yes,1,86,3239,Competitor made better offer
1,9237-HQITU,1,United States,California,Los Angeles,90005,"34.059281, -118.30742",34.059281,-118.307420,Female,...,Month-to-month,Yes,Electronic check,70.70,151.65,Yes,1,67,2701,Moved
2,9305-CDSKC,1,United States,California,Los Angeles,90006,"34.048013, -118.293953",34.048013,-118.293953,Female,...,Month-to-month,Yes,Electronic check,99.65,820.5,Yes,1,86,5372,Moved
3,7892-POOKP,1,United States,California,Los Angeles,90010,"34.062125, -118.315709",34.062125,-118.315709,Female,...,Month-to-month,Yes,Electronic check,104.80,3046.05,Yes,1,84,5003,Moved
4,0280-XJGEX,1,United States,California,Los Angeles,90015,"34.039224, -118.266293",34.039224,-118.266293,Male,...,Month-to-month,Yes,Bank transfer (automatic),103.70,5036.3,Yes,1,89,5340,Competitor had better devices


# 2. Standardize column names (snake_case)

In [4]:
df.columns = [re.sub(r"[^0-9a-z]+", "_", c.strip().lower()).strip("_") for c in df.columns]
df = df.rename(columns={"customerid": "customer_id"})

In [ ]:
df["total_charges"] = pd.to_numeric(df["total_charges"], errors="coerce")

churn_reason    5174
dtype: int64

# 3. Check duplicates and missing values

In [6]:
print("Duplicate customer IDs:", df["customer_id"].duplicated().sum())
print("Missing values per column:")
print(df.isna().sum()[df.isna().sum() > 0])

Duplicate customer IDs: 0
Missing values per column:
churn_reason    5174
dtype: int64


# 4. Fix total_charges (stored as text has blank values)

In [20]:
df["total_charges"] = pd.to_numeric(df["total_charges"], errors="coerce")
blank_rows = df["total_charges"].isna()
print("Blank total_charges rows:", blank_rows.sum())
print("Their tenure_months values:", df.loc[blank_rows, "tenure_months"].unique())
df.loc[blank_rows & (df["tenure_months"] == 0), "total_charges"] = 0

Blank total_charges rows: 0
Their tenure_months values: []


# 5. Drop columns with no analytical value

In [21]:
constant_cols = [c for c in df.columns if df[c].nunique() == 1]
print("Constant columns dropped:", constant_cols)
df = df.drop(columns=constant_cols)

Constant columns dropped: []


# 6. Clean Categorical values

In [10]:
service_cols = [
    "multiple_lines", "online_security", "online_backup", "device_protection", "tech_support", "streaming_tv", "streaming_movies"
]

for col in service_cols:
    # "No internet service" / "No phone service" are effectively "No"
    df[col] = df[col].replace({"No internet service": "No", "No phone servie": "No"})

for col in df.select_dtypes(include="object").columns:
    df[col] = df[col].str.strip()

# 7. Future engineering

In [11]:
df["tenurebucket"] = pd.cut(
    df["tenure_months"],
    bins=[-1, 12, 24, 48, 72],
    labels=["0-12", "13-24", "25-48", "49-72"]
).astype(str)

df["numservices"] = (df[service_cols[1:]] == "Yes").sum(axis=1) # executes multiplelines

# 8. Validate

In [13]:
assert df["customer_id"].is_unique
assert df["total_charges"].notna().all()
assert set(df["churn_value"].unique()) <= {0,1}
print("Overal churn rate: {:.1f}%".format(df["churn_value"].mean() * 100))

Overal churn rate: 26.5%


# 9. Save

In [14]:
df.to_csv(CLEAN_CSV, index=False)
with sqlite3.connect(DB) as conn:
    df.to_sql("cumtomers", conn, if_exists="replace", index=False)

print(f"Saved {len(df)} rows to {CLEAN_CSV} and table 'customers' in {DB}")

Saved 7043 rows to D:/My Resume/IBM_TELECOM/telcom/telco_clean.csv and table 'customers' in D:/My Resume/IBM_TELECOM/telcom/churn.db


In [15]:
df["payment_method"]

0                    Mailed check
1                Electronic check
2                Electronic check
3                Electronic check
4       Bank transfer (automatic)
                  ...            
7038    Bank transfer (automatic)
7039                 Mailed check
7040      Credit card (automatic)
7041             Electronic check
7042    Bank transfer (automatic)
Name: payment_method, Length: 7043, dtype: object